# R2/R3/R4 해당 생성 (Qwen2.5-7B-Instruct, 4bit)

**실행 전 준비물** (Google Drive `MyDrive/ade-project/`에 업로드):
- `unified.jsonl` (로컬 `research-project/data/unified.jsonl` — 원문이 포함되어 gitignore 대상이라 git clone에 안 따라옴)

**주의**: 이 노트북은 학생 모델 학습(train_qlora.ipynb)과 같은 세션에서 돌리지 말 것 (VRAM 부족). 끝나면 런타임을 재시작해 VRAM을 비우고 다음 단계로 넘어갈 것.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ade-project'
import os
assert os.path.exists(f'{DRIVE_DIR}/unified.jsonl'), (
    f'{DRIVE_DIR}/unified.jsonl 없음 — 로컬 research-project/data/unified.jsonl을 '
    'Drive의 이 경로에 먼저 업로드할 것')

In [ ]:
REPO_URL = 'https://github.com/Gaeul5/Oracle_healthcare-bio_sLLM.git'

import os
if not os.path.exists('/content/repo'):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo/research-project

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## 1. 소량으로 속도/VRAM 확인 (먼저 실행)

전체 10,340건 × 3조건을 바로 돌리기 전에, 20건만으로 속도를 재고 전체 소요 시간을 가능해보자.

In [ ]:
!python src/generate_teacher_data.py \
    --unified {DRIVE_DIR}/unified.jsonl \
    --out {DRIVE_DIR}/teacher_outputs.jsonl \
    --checkpoint-every 5 \
    --limit 20

위 출력의 "남은 예상 N분"을 보고 전체(10,320건 남음) 소요 시간을 가능해볼 것. 자유 Colab은 세션 시간 제한이 있으니, 오래 걸릴 것 같으면 여러 세션에 걸쳐 아래 셀을 반복 실행하면 된다 (이미 완료된 doc_id는 자동으로 건너뜀).

## 2. 전체 실행 (여러 세션에 걸쳐 재실행 가능 — 이미 완료된 doc_id는 건너뜀)

In [ ]:
!python src/generate_teacher_data.py \
    --unified {DRIVE_DIR}/unified.jsonl \
    --out {DRIVE_DIR}/teacher_outputs.jsonl \
    --checkpoint-every 20